# Fig 3.6.1 — Combined parameter fit score

Standalone figure: combined log-ratio fit score across the $E \times \sigma_\text{vol}$ grid.
Loads cached results from fig3_1_1 and fig3_4_1 — no simulations are rerun.

| Panel | Content |
|-------|---------|
| **A** | Combined log-ratio score (PVC + subspace overlap MSEs) |

In [7]:
%matplotlib inline
import os, sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('.'))
from plot_style import apply_style, COLORS, FIG_SIZES, FONT, LW, LINEWIDTH_IN

NOTEBOOK_DIR = os.path.abspath('')
DATA_DIR     = os.path.join(NOTEBOOK_DIR, 'data')
FIGURES_DIR  = os.path.join(NOTEBOOK_DIR, 'saved_figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

# Must match both sweep grids exactly
GRID_E       = [0.0, 0.25, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
GRID_VOL_STD = [0.0, 0.001, 0.002, 0.005, 0.01, 0.015, 0.02, 0.03]
DRIFT_THRESH = 0.90   # day-7 overlap above this = no meaningful drift
DAY_MAX      = 7
MOUSE_IDS    = ['m01', 'm03', 'm04', 'm05']

GRID_CACHE     = os.path.join(DATA_DIR, 'grid_search_cache.npz')
SUBSPACE_CACHE = os.path.join(DATA_DIR, 'subspace_overlap_cache.npz')
RULE_OV_CACHE  = os.path.join(DATA_DIR, 'rule_subspace_overlap_cache.npz')

print('Data dir:', DATA_DIR)
for p in [GRID_CACHE, SUBSPACE_CACHE, RULE_OV_CACHE]:
    print(f'  {os.path.basename(p)}: {"found" if os.path.isfile(p) else "MISSING"}')

Data dir: c:\Users\sctcl\Thesis_Exct&Vol_Model\results_figures\data
  grid_search_cache.npz: found
  subspace_overlap_cache.npz: found
  rule_subspace_overlap_cache.npz: found


In [8]:
# ── Load PVC grid search cache ────────────────────────────────────────────────
with np.load(GRID_CACHE) as _gc:
    mse_pvc = _gc['loss_grid'].copy()
    gc_E    = [float(v) for v in _gc['GRID_E']]
    gc_vol  = [float(v) for v in _gc['GRID_VOL_STD']]

assert gc_E == GRID_E and gc_vol == GRID_VOL_STD, \
    f'PVC cache grid mismatch: {gc_E} vs {GRID_E}'

# ── Load subspace overlap sweep cache ────────────────────────────────────────
with np.load(SUBSPACE_CACHE) as _sw:
    ov_final_grid = _sw['ov_final'].copy()
    ov_traj_grid  = _sw['ov_traj'].copy()
    sw_E          = [float(v) for v in _sw['GRID_E']]
    sw_vol        = [float(v) for v in _sw['GRID_VOL_STD']]

assert sw_E == GRID_E and sw_vol == GRID_VOL_STD, \
    f'Subspace cache grid mismatch: {sw_E} vs {GRID_E}'

# ── Pool Rule et al. subspace overlap by day (for MSE target) ─────────────────
with np.load(RULE_OV_CACHE) as _ro:
    rule_ov_raw = {
        mid: dict(days=_ro[f'{mid}_days'].copy(), ov=_ro[f'{mid}_ov'].copy())
        if f'{mid}_days' in _ro else None
        for mid in MOUSE_IDS
    }

rule_ov_by_day = {d: [] for d in range(2, DAY_MAX + 1)}
for mid, data in rule_ov_raw.items():
    if data is None:
        continue
    for day, ov in zip(data['days'], data['ov']):
        d = int(round(day))
        if 2 <= d <= DAY_MAX:
            rule_ov_by_day[d].append(ov)

rule_ov_mean = {
    d: float(np.mean(vals)) if vals else np.nan
    for d, vals in rule_ov_by_day.items()
}

# ── Compute subspace overlap MSE (no simulations) ────────────────────────────
comp_days   = [d for d in range(2, DAY_MAX + 1)
               if not np.isnan(rule_ov_mean.get(d, np.nan))]
nE, nV      = len(GRID_E), len(GRID_VOL_STD)
ov_mse_grid = np.full((nE, nV), np.nan)

for ii in range(nE):
    for jj in range(nV):
        traj = ov_traj_grid[ii, jj]
        sq   = [(traj[d - 1] - rule_ov_mean[d]) ** 2
                for d in comp_days
                if not (np.isnan(traj[d - 1]) or np.isnan(rule_ov_mean[d]))]
        if sq:
            ov_mse_grid[ii, jj] = float(np.mean(sq))

invalid_mask = ov_final_grid >= DRIFT_THRESH

print('Caches loaded.')
print(f'PVC grid:  {nE}x{nV},  NaN={int(np.isnan(mse_pvc).sum())}')
print(f'OV grid:   {nE}x{nV},  NaN={int(np.isnan(ov_mse_grid).sum())}')
print(f'Hatched (no-drift): {int(invalid_mask.sum())}/{invalid_mask.size} cells')
best_pvc = np.unravel_index(np.nanargmin(np.where(invalid_mask, np.nan, mse_pvc)), mse_pvc.shape)
best_ov  = np.unravel_index(np.nanargmin(np.where(invalid_mask, np.nan, ov_mse_grid)), ov_mse_grid.shape)
print(f'Best PVC fit (valid): E={GRID_E[best_pvc[0]]}  vol={GRID_VOL_STD[best_pvc[1]]}')
print(f'Best OV fit  (valid): E={GRID_E[best_ov[0]]}   vol={GRID_VOL_STD[best_ov[1]]}')

Caches loaded.
PVC grid:  8x8,  NaN=0
OV grid:   8x8,  NaN=0
Hatched (no-drift): 0/64 cells
Best PVC fit (valid): E=0.5  vol=0.01
Best OV fit  (valid): E=1.0   vol=0.005


In [9]:
# ── Plot helpers ──────────────────────────────────────────────────────────────
apply_style()
import matplotlib as _mpl

DATA_COLOR = '#888888'


def _heatmap(ax, grid, cmap_name, vmin, vmax, cbar_label, title, letter):
    # NaN cells render as light grey (distinct from both best-fit dark and no-drift hatch)
    cmap = _mpl.colormaps[cmap_name].copy()
    cmap.set_bad('lightgrey')
    im = ax.imshow(grid, origin='lower', aspect='auto', cmap=cmap,
                   vmin=vmin, vmax=vmax,
                   extent=[-0.5, nV - 0.5, -0.5, nE - 0.5])
    ax.set_xticks(range(nV))
    ax.set_xticklabels([str(v) for v in GRID_VOL_STD], rotation=45, ha='right',
                        fontsize=FONT.get('annotation', 8) - 1)
    ax.set_yticks(range(nE))
    ax.set_yticklabels([str(e) for e in GRID_E],
                        fontsize=FONT.get('annotation', 8) - 1)
    ax.set_xlabel('Synaptic volatility (vol_std)')
    ax.set_ylabel('Excitability amplitude (E)')
    ax.set_title(title)
    cb = ax.figure.colorbar(im, ax=ax)
    cb.set_label(cbar_label, fontsize=FONT.get('annotation', 8) - 1)
    ax.text(-0.12, 1.02, letter, transform=ax.transAxes,
            fontsize=FONT.get('panel_label', 13), fontweight='bold')
    return im


def _hatch(ax, mask):
    for ii in range(nE):
        for jj in range(nV):
            if mask[ii, jj]:
                rect = plt.Rectangle(
                    [jj - 0.5, ii - 0.5], 1, 1,
                    fill=True, facecolor='lightgrey', alpha=0.55,
                    hatch='///', edgecolor='grey', linewidth=0.4, zorder=3,
                )
                ax.add_patch(rect)
    ax.plot([], [], color='grey', lw=0, marker='s',
            markerfacecolor='lightgrey', markeredgecolor='grey', markersize=9,
            label=f'No-drift (overlap ≥ {DRIFT_THRESH} at day 7)')


def _star(ax, score_grid, minimize=True):
    g    = np.where(invalid_mask, np.nan, score_grid)
    fn   = np.nanargmin if minimize else np.nanargmax
    best = np.unravel_index(fn(g), g.shape)
    ax.plot(best[1], best[0], 'r*', markersize=14,
            label=f'Best: E={GRID_E[best[0]]}, vol={GRID_VOL_STD[best[1]]}')
    ax.legend(fontsize=FONT.get('annotation', 8) - 1, loc='upper right')


def _log_ratio(x):
    mn = np.nanmin(x)
    return np.log(np.clip(x / mn, 1.0, None))


print('Helpers ready.')

Helpers ready.


In [ ]:
# ── Fig 3.6.1 — Combined log-ratio parameter fit score ───────────────────────
# Figure sized to match one heatmap panel from fig3_7_1 (LINEWIDTH_IN x 6.5, 2x2 grid).
# width_ratios=[1,2,1] centres the heatmap at ~50% figure width with equal whitespace.
import matplotlib.gridspec as gridspec

CANONICAL_E   = 0.5
CANONICAL_VOL = 0.01

apply_style()

fig = plt.figure(figsize=(LINEWIDTH_IN, 3.5))
gs  = gridspec.GridSpec(1, 3, width_ratios=[1, 2, 1], figure=fig)
ax  = fig.add_subplot(gs[0, 1])

combined      = _log_ratio(mse_pvc) + _log_ratio(ov_mse_grid)
combined_norm = combined / np.nanmax(combined)

_heatmap(ax, combined_norm, 'viridis', 0.0, 1.0,
         'Normalised log-ratio score',
         'Combined fit score', '')
_star(ax, combined, minimize=True)

if CANONICAL_E in GRID_E and CANONICAL_VOL in GRID_VOL_STD:
    ci = GRID_E.index(CANONICAL_E)
    cj = GRID_VOL_STD.index(CANONICAL_VOL)
    ax.plot(cj, ci, 'w*', markersize=10,
            markeredgecolor='black', markeredgewidth=0.5,
            label=f'Model: E={CANONICAL_E}, σ={CANONICAL_VOL}')
    ax.legend(fontsize=FONT.get('annotation', 8) - 1, loc='upper right')

best_c = np.unravel_index(
    np.nanargmin(np.where(invalid_mask, np.nan, combined)), combined.shape)
print(f'Best valid fit: E={GRID_E[best_c[0]]}  '
      f'vol_std={GRID_VOL_STD[best_c[1]]}  log-score={combined[best_c]:.3f}')

plt.tight_layout()
out = os.path.join(FIGURES_DIR, 'fig3_6_1_parameter_selection.pdf')
fig.savefig(out, dpi=300, bbox_inches='tight')
print('Saved to', out)
plt.show()